In [57]:
import pandas as pd

# 1. CSV-Datei einlesen
df = pd.read_csv("C:/Users/pazi9/OneDrive/Desktop/shiny2/shiny_visualisation/detailed_car_sales_data_all.csv")



In [58]:

# 2. Einzigartige Kombinationen von 'manufacturer' und 'model' extrahieren
unique_keys = df[['manufacturer', 'model']].drop_duplicates().reset_index(drop=True)

# 3. In eine JSON-Datei speichern
unique_keys.to_json('model_keys.json', orient='records', indent=4)

print("JSON-Datei 'model_keys.json' wurde erstellt.")
print("Inhalt der extrahierten Daten:")
print(unique_keys)

JSON-Datei 'model_keys.json' wurde erstellt.
Inhalt der extrahierten Daten:
    manufacturer    model
0          Acura      MDX
1          Acura      RDX
2          Acura      TLX
3          Acura      TSX
4          Acura      ILX
..           ...      ...
685   Volkswagen      Bus
686   Volkswagen    Civic
687   Volkswagen  Corrado
688   Volkswagen     GOLF
689   Volkswagen   Cabrio

[690 rows x 2 columns]


In [59]:
# To install required packages, run:
!pip install icrawler pandas
!pip install --upgrade icrawler

In [60]:
!pip install Pillow
!pip install imagehash


In [64]:
import pandas as pd
import os
import time  # WIEDER HINZUGEFÜGT
import hashlib
from icrawler.builtin import BingImageCrawler  # ZURÜCK ZU BING
from PIL import Image
import imagehash

# --- Konfiguration ---
JSON_DATEI = 'model_keys.json'
BILDER_ORDNER = 'auto_bilder'
TARGET_WIDTH = 1280
TARGET_HEIGHT = 720
MAX_IMAGES_TO_CHECK = 10  # WIEDER AUF 10 (oder höher)
PHASH_DISTANCE_THRESHOLD = 5
# ---------------------

def get_image_phash(image_path):
    """Erstellt einen perzeptuellen Hash (pHash) des Bildes."""
    try:
        with Image.open(image_path) as img:
            img = img.convert("RGB") 
            return imagehash.phash(img)
    except Exception as e:
        print(f"Fehler beim Hashen von {image_path}: {e}")
        return None

print(f"Lese JSON-Datei: {JSON_DATEI}")

try:
    df = pd.read_json(JSON_DATEI, orient='records')
except FileNotFoundError:
    print(f"FEHLER: Die Datei '{JSON_DATEI}' wurde nicht gefunden.")
    exit()

os.makedirs(BILDER_ORDNER, exist_ok=True)
seen_image_hashes = []
failed_models = []

print("Prüfe existierende Bilder, um Duplikate zu vermeiden...")
for root, _, files in os.walk(BILDER_ORDNER):
    for file in files:
        if file.endswith('.jpg'):
            full_path = os.path.join(root, file)
            existing_hash = get_image_phash(full_path)
            if existing_hash:
                seen_image_hashes.append(existing_hash)
print(f"{len(seen_image_hashes)} bereits existierende Bild-Hashes geladen.")

# --- Hauptschleife ---
for index, row in df.iterrows():
    manufacturer = row['manufacturer']
    model = row['model']
    
    print(f"\n--- Verarbeite: {manufacturer} {model} ---")

    manufacturer_dir = os.path.join(BILDER_ORDNER, manufacturer)
    os.makedirs(manufacturer_dir, exist_ok=True)

    target_filename = f"{model}.jpg"
    target_path = os.path.join(manufacturer_dir, target_filename)

    if os.path.exists(target_path):
        print(f"Endgültiges Bild '{target_filename}' existiert bereits. Überspringe.")
        continue

    search_query = f"{model}+{manufacturer}+{manufacturer}+{model}+car+photo+high+resolution+exterior+view+side+front+angle+studio+lighting+clear+background+detailed+shot+professional+automobile+image+HD+quality+4K+sharp+focus+clean+look+modern+design+sleek+stylish+automotive+vehicle"
    
    crawler = BingImageCrawler(
        storage={'root_dir': manufacturer_dir}
    )
    
    found_unique_image_for_model = False
    
    try:
        print(f"Suche (Bing) nach: '{search_query}' (Filter: 'photo', max: {MAX_IMAGES_TO_CHECK})...")
        crawler.crawl(
            keyword=search_query,
            filters=dict(
                size='large',
                type='photo'
            ),
            max_num=MAX_IMAGES_TO_CHECK
        )
        
        for i in range(1, MAX_IMAGES_TO_CHECK + 1):
            temp_download_file = os.path.join(manufacturer_dir, f'{i:06d}.jpg')

            if not os.path.exists(temp_download_file):
                continue

            print(f"Prüfe Kandidat: {temp_download_file}")
            current_hash = get_image_phash(temp_download_file)
            
            if current_hash is None:
                print("  -> Fehler: Bild-Hash konnte nicht erstellt werden. Überspringe.")
                continue

            is_duplicate = False
            for seen_hash in seen_image_hashes:
                distance = current_hash - seen_hash
                if distance < PHASH_DISTANCE_THRESHOLD:
                    print(f"  -> Visuelles Duplikat gefunden (Distanz: {distance}). Versuche nächstes Bild.")
                    is_duplicate = True
                    break
            
            if is_duplicate:
                continue
            
            print(f"  -> Einzigartiges Bild gefunden (Hash: {current_hash}). Verarbeite...")
            seen_image_hashes.append(current_hash)
            
            try:
                # Bildbearbeitung
                img = Image.open(temp_download_file)
                original_width, original_height = img.size
                target_aspect = TARGET_WIDTH / TARGET_HEIGHT
                original_aspect = original_width / original_height

                if original_aspect > target_aspect:
                    new_height = original_height; new_width = int(new_height * target_aspect)
                    left = (original_width - new_width) / 2; top = 0
                    right = (original_width + new_width) / 2; bottom = original_height
                else:
                    new_width = original_width; new_height = int(new_width / target_aspect)
                    left = 0; top = (original_height - new_height) / 2
                    right = original_width; bottom = (original_height + new_height) / 2
                
                img = img.crop((left, top, right, bottom))
                img = img.resize((TARGET_WIDTH, TARGET_HEIGHT), Image.LANCZOS)
                if img.mode != 'RGB': img = img.convert('RGB')
                
                img.save(target_path)
                print(f"Erfolgreich bearbeitetes und gespeichertes Bild: '{target_path}'")
                found_unique_image_for_model = True
                break
                
            except Exception as img_e:
                print(f"FEHLER bei der Bildbearbeitung von '{temp_download_file}': {img_e}")

        if not found_unique_image_for_model:
            print(f"Fehler: Kein einzigartiges Bild (aus {MAX_IMAGES_TO_CHECK} Versuchen) für {manufacturer} {model} gefunden.")
            failed_models.append(f"{manufacturer} {model}")

    except Exception as e:
        print(f"Ein schwerwiegender Fehler ist aufgetreten bei {manufacturer} {model}: {e}")
        failed_models.append(f"{manufacturer} {model} (Crawler-Fehler)")
            
    finally:
        # Aufräumen
        for i in range(1, MAX_IMAGES_TO_CHECK + 1):
            temp_download_file = os.path.join(manufacturer_dir, f'{i:06d}.jpg')
            if os.path.exists(temp_download_file):
                try: os.remove(temp_download_file)
                except Exception: pass

    # time.sleep(1) # WIEDER HINZUGEFÜGT - Sehr wichtig!

# --- Skript-Ende ---
print("\nDownload- und Bearbeitungsvorgang abgeschlossen.")
print(f"Insgesamt {len(seen_image_hashes)} einzigartige Bilder sind jetzt im Set.")

if failed_models:
    print(f"\n{len(failed_models)} Modell(e) konnten nicht verarbeitet werden.")
    with open('fehlgeschlagene_modelle.txt', 'w', encoding='utf-8') as f:
        f.write("Folgende Modelle konnten nicht automatisch heruntergeladen werden (Duplikate oder Fehler):\n")
        for item in failed_models:
            f.write(f"- {item}\n")
    print("Eine Liste wurde in 'fehlgeschlagene_modelle.txt' gespeichert.")
else:
    print("\nAlle Modelle wurden erfolgreich verarbeitet.")

Lese JSON-Datei: model_keys.json
Prüfe existierende Bilder, um Duplikate zu vermeiden...


2025-11-18 10:14:37,821 - INFO - icrawler.crawler - start crawling...
2025-11-18 10:14:37,821 - INFO - icrawler.crawler - starting 1 feeder threads...
2025-11-18 10:14:37,823 - INFO - feeder - thread feeder-001 exit
2025-11-18 10:14:37,826 - INFO - icrawler.crawler - starting 1 parser threads...
2025-11-18 10:14:37,828 - INFO - icrawler.crawler - starting 1 downloader threads...


667 bereits existierende Bild-Hashes geladen.

--- Verarbeite: Acura MDX ---
Endgültiges Bild 'MDX.jpg' existiert bereits. Überspringe.

--- Verarbeite: Acura RDX ---
Endgültiges Bild 'RDX.jpg' existiert bereits. Überspringe.

--- Verarbeite: Acura TLX ---
Endgültiges Bild 'TLX.jpg' existiert bereits. Überspringe.

--- Verarbeite: Acura TSX ---
Endgültiges Bild 'TSX.jpg' existiert bereits. Überspringe.

--- Verarbeite: Acura ILX ---
Endgültiges Bild 'ILX.jpg' existiert bereits. Überspringe.

--- Verarbeite: Acura RLX ---
Endgültiges Bild 'RLX.jpg' existiert bereits. Überspringe.

--- Verarbeite: Acura TL ---
Endgültiges Bild 'TL.jpg' existiert bereits. Überspringe.

--- Verarbeite: Acura EL ---
Endgültiges Bild 'EL.jpg' existiert bereits. Überspringe.

--- Verarbeite: Acura RSX ---
Endgültiges Bild 'RSX.jpg' existiert bereits. Überspringe.

--- Verarbeite: Acura NSX ---
Endgültiges Bild 'NSX.jpg' existiert bereits. Überspringe.

--- Verarbeite: Acura CSX ---
Endgültiges Bild 'CSX.jpg' 

2025-11-18 10:14:38,446 - INFO - parser - parsing result page https://www.bing.com/images/async?q=230+BMW+BMW+230+car+photo+high+resolution+exterior+view+side+front+angle+studio+lighting+clear+background+detailed+shot+professional+automobile+image+HD+quality+4K+sharp+focus+clean+look+modern+design+sleek+stylish+automotive+vehicle&first=0&qft=+filterui:imagesize-large+filterui:photo-photo
2025-11-18 10:14:40,772 - INFO - downloader - image #1	https://content.pexels.com/images/canva/ai-generated-ad/off-theme/forest_starry_winter_night-full.jpg
2025-11-18 10:14:41,457 - INFO - downloader - image #2	https://static.car.gr/340800141_e_z.jpg
2025-11-18 10:14:42,013 - INFO - downloader - image #3	https://static.car.gr/42583103_s_z.jpg
2025-11-18 10:14:43,458 - INFO - downloader - image #4	https://media.hatla2eestatic.com/uploads/car/2025/06/01/6986436/full_up_9ccdc170cae1be5b042a0d955d215857.jpg
2025-11-18 10:14:44,973 - INFO - downloader - image #5	https://media.hatla2eestatic.com/uploads/car

Prüfe Kandidat: auto_bilder\BMW\000001.jpg
  -> Visuelles Duplikat gefunden (Distanz: 2). Versuche nächstes Bild.
Prüfe Kandidat: auto_bilder\BMW\000002.jpg
  -> Einzigartiges Bild gefunden (Hash: c68f78912a8fad2a). Verarbeite...
Erfolgreich bearbeitetes und gespeichertes Bild: 'auto_bilder\BMW\230.jpg'

--- Verarbeite: BMW M3 ---
Endgültiges Bild 'M3.jpg' existiert bereits. Überspringe.

--- Verarbeite: BMW 428i ---
Endgültiges Bild '428i.jpg' existiert bereits. Überspringe.

--- Verarbeite: BMW 535i ---
Endgültiges Bild '535i.jpg' existiert bereits. Überspringe.

--- Verarbeite: BMW 540 ---
Endgültiges Bild '540.jpg' existiert bereits. Überspringe.

--- Verarbeite: BMW 540I ---
Endgültiges Bild '540I.jpg' existiert bereits. Überspringe.

--- Verarbeite: BMW 530I ---
Endgültiges Bild '530I.jpg' existiert bereits. Überspringe.

--- Verarbeite: BMW 7 ---
Endgültiges Bild '7.jpg' existiert bereits. Überspringe.

--- Verarbeite: BMW 340i ---
Endgültiges Bild '340i.jpg' existiert bereits. 

2025-11-18 10:14:58,865 - INFO - parser - parsing result page https://www.bing.com/images/async?q=CT5-V+Cadillac+Cadillac+CT5-V+car+photo+high+resolution+exterior+view+side+front+angle+studio+lighting+clear+background+detailed+shot+professional+automobile+image+HD+quality+4K+sharp+focus+clean+look+modern+design+sleek+stylish+automotive+vehicle&first=0&qft=+filterui:imagesize-large+filterui:photo-photo
2025-11-18 10:15:03,638 - INFO - downloader - image #1	https://www.hdcarwallpapers.com/download/cadillac_ct5_v_2025-1920x1080.jpg
2025-11-18 10:15:13,160 - INFO - downloader - image #2	https://cadillacsociety.com/wp-content/uploads/2024/06/2025-Cadillac-CT5-V-Blackwing-Drift-Metallic-GAE-Live-Photo-Exterior-001.jpg
2025-11-18 10:15:14,545 - INFO - downloader - image #3	https://www.contecadillac.com/static/dealer-19757/Paul-Conte-Cadillac_CT5-V-Blackwing_ls3.jpg
2025-11-18 10:15:18,176 - INFO - downloader - image #4	https://cadillacsociety.com/wp-content/uploads/2024/01/2025-Cadillac-CT5-V

Prüfe Kandidat: auto_bilder\Cadillac\000001.jpg
  -> Visuelles Duplikat gefunden (Distanz: 0). Versuche nächstes Bild.
Prüfe Kandidat: auto_bilder\Cadillac\000002.jpg
  -> Einzigartiges Bild gefunden (Hash: ad9df0f08287ce64). Verarbeite...
Erfolgreich bearbeitetes und gespeichertes Bild: 'auto_bilder\Cadillac\CT5-V.jpg'

--- Verarbeite: Cadillac CT4-V ---
Endgültiges Bild 'CT4-V.jpg' existiert bereits. Überspringe.

--- Verarbeite: Cadillac Eldorado ---
Endgültiges Bild 'Eldorado.jpg' existiert bereits. Überspringe.

--- Verarbeite: Cadillac DeVille ---
Endgültiges Bild 'DeVille.jpg' existiert bereits. Überspringe.

--- Verarbeite: Cadillac ATS-V ---
Endgültiges Bild 'ATS-V.jpg' existiert bereits. Überspringe.

--- Verarbeite: Chevrolet Corvette ---
Endgültiges Bild 'Corvette.jpg' existiert bereits. Überspringe.

--- Verarbeite: Chevrolet Impala ---
Endgültiges Bild 'Impala.jpg' existiert bereits. Überspringe.

--- Verarbeite: Chevrolet Cruze ---
Endgültiges Bild 'Cruze.jpg' existiert 